# Part 1.1: Piecewise KNN Regression

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.spatial.distance import cdist

## Data Preprocessing

In [ ]:
# Load data
wb_df = pd.read_csv('data/worldbank_2017_2021.csv')
regions_df = pd.read_excel('data/countries_regions.xlsx')

# Name mapping
name_map = {
    'United States': 'United States of America',
    'Viet Nam': 'Vietnam'
}
wb_df['Country Name'] = wb_df['Country Name'].replace(name_map)

# Drop columns with >50% missing
missing_pct = wb_df.isna().mean()
wb_df = wb_df.drop(columns=missing_pct[missing_pct > 0.5].index.tolist())

# Numeric columns
num_cols = [c for c in wb_df.select_dtypes(include=['float64', 'int64']).columns if c != 'Year']

# Fill missing with median + clip outliers
for col in num_cols:
    wb_df[col] = wb_df[col].fillna(wb_df[col].median())
    wb_df[col] = wb_df[col].clip(wb_df[col].quantile(0.01), wb_df[col].quantile(0.99))

# Merge with regions
df = wb_df.merge(regions_df, left_on='Country Name', right_on='CountryName', how='inner').drop(columns=['CountryName'])

target_col = 'GNI per capita, PPP (current international $)'
df = df.dropna(subset=[target_col])
feature_cols = [c for c in num_cols if c != target_col]

In [ ]:
# Split test using year
train_df = df[df['Year'] < 2021].copy()
test_df = df[df['Year'] == 2021].copy()

y_train = train_df[target_col].values
y_test = test_df[target_col].values

print(f'Train (2017-2020): {len(train_df)}')
print(f'Test (2021): {len(test_df)}')
print(f'Features: {len(feature_cols)}')

---

# Baseline: Correlation + Lasso + Random Forest

Same as Project 1 part2.2

## Feature Selection

### Correlation Filter

In [ ]:
corr = train_df[feature_cols].corrwith(train_df[target_col]).abs()
w2_corr_features = corr[corr >= 0.3].index.tolist()
print(f'Correlation filter (>=0.3): {len(w2_corr_features)} features')

### Lasso Feature Selection

In [ ]:
X_train_corr = train_df[w2_corr_features].values
scaler_corr = StandardScaler()
X_train_corr_scaled = scaler_corr.fit_transform(X_train_corr)

lasso_fs = Lasso(alpha=100, max_iter=10000)
lasso_fs.fit(X_train_corr_scaled, y_train)
w2_lasso_features = [f for f, c in zip(w2_corr_features, lasso_fs.coef_) if c != 0]
print(f'Lasso selection: {len(w2_lasso_features)} features')

## Modeling

### Random Forest

In [ ]:
X_train_bl = train_df[w2_lasso_features].values
X_test_bl = test_df[w2_lasso_features].values
scaler_bl = StandardScaler()
X_train_bl_scaled = scaler_bl.fit_transform(X_train_bl)
X_test_bl_scaled = scaler_bl.transform(X_test_bl)

rf_baseline = RandomForestRegressor(n_estimators=100, random_state=23)
rf_baseline.fit(X_train_bl_scaled, y_train)
y_pred_baseline = rf_baseline.predict(X_test_bl_scaled)

## Evaluation

In [ ]:
bl_results = {
    'R2': r2_score(y_test, y_pred_baseline),
    'RMSE': root_mean_squared_error(y_test, y_pred_baseline),
    'MAE': mean_absolute_error(y_test, y_pred_baseline)
}

print(f'R2:   {bl_results["R2"]:.6f}')
print(f'RMSE: {bl_results["RMSE"]:.2f}')
print(f'MAE:  {bl_results["MAE"]:.2f}')

---

# Single KNN (Tuned)

Using same Corr+Lasso features as baseline, but KNN instead of RF

## Modeling

### KNN with GridSearchCV

In [ ]:
knn_single_gs = GridSearchCV(
    KNeighborsRegressor(),
    {
        'n_neighbors': [3, 5, 7, 9, 11, 15, 20],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    },
    cv=5, scoring='r2', n_jobs=-1
)
knn_single_gs.fit(X_train_bl_scaled, y_train)
y_pred_single_knn = knn_single_gs.predict(X_test_bl_scaled)

print(f'Best params: {knn_single_gs.best_params_}')
print(f'CV R2: {knn_single_gs.best_score_:.4f}')

## Evaluation

In [ ]:
single_knn_results = {
    'R2': r2_score(y_test, y_pred_single_knn),
    'RMSE': root_mean_squared_error(y_test, y_pred_single_knn),
    'MAE': mean_absolute_error(y_test, y_pred_single_knn)
}

print(f'R2:   {single_knn_results["R2"]:.6f}')
print(f'RMSE: {single_knn_results["RMSE"]:.2f}')
print(f'MAE:  {single_knn_results["MAE"]:.2f}')

---

# Piecewise KNN Regression

## Clustering Feature Selection

In [ ]:
country_avg = train_df[['Country Name'] + feature_cols].groupby('Country Name').mean().reset_index()
X_country = country_avg[feature_cols].copy()

# VarianceThreshold
scaler_vt = StandardScaler()
X_scaled_vt = pd.DataFrame(scaler_vt.fit_transform(X_country), columns=feature_cols, index=X_country.index)
vt_selector = VarianceThreshold(threshold=0.01)
vt_selector.fit(X_scaled_vt)
vt_kept = X_scaled_vt.columns[vt_selector.get_support()].tolist()

# Remove highly correlated features (>0.9)
X_vt = X_scaled_vt[vt_kept]
corr_matrix = X_vt.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
cluster_features = [c for c in vt_kept if c not in to_drop]

# Scale for clustering
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_country[cluster_features])

print(f'VarianceThreshold kept: {len(vt_kept)}')
print(f'After corr>0.9 filter: {len(cluster_features)}')

## Determine k

In [ ]:
# Elbow + Silhouette
wcss = []
sil_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=23)
    labels = kmeans.fit_predict(X_cluster_scaled)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_range, wcss, marker='o')
axes[0].set_xlabel('K'); axes[0].set_ylabel('WCSS'); axes[0].set_title('Elbow Method')
axes[1].plot(k_range, sil_scores, marker='o', color='orange')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('Silhouette Score')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation metrics for each k
def dunn_index(X, labels):
    unique_clusters = np.unique(labels)
    centroids = np.array([X[labels == k].mean(axis=0) for k in unique_clusters])
    inter_cluster_dist = np.min(
        cdist(centroids, centroids)[np.triu_indices(len(unique_clusters), k=1)]
    )
    intra_cluster_dist = max(
        np.max(cdist(X[labels == k], X[labels == k]))
        for k in unique_clusters
    )
    return inter_cluster_dist / intra_cluster_dist

def evaluate_clustering(X, X_clusters):
    silhouette = silhouette_score(X, X_clusters)
    db = davies_bouldin_score(X, X_clusters)
    dunn = dunn_index(X, X_clusters)
    ch = calinski_harabasz_score(X, X_clusters)
    return silhouette, db, dunn, ch

eval_rows = []
for k in k_range:
    km_tmp = KMeans(n_clusters=k, n_init=10, random_state=23)
    lbl_tmp = km_tmp.fit_predict(X_cluster_scaled)
    s, d, du, c = evaluate_clustering(X_cluster_scaled, lbl_tmp)
    eval_rows.append({'k': k, 'Silhouette': round(s, 4), 'Davies-Bouldin': round(d, 4),
                      'Dunn': round(du, 4), 'Calinski-Harabasz': round(c, 2)})

eval_df = pd.DataFrame(eval_rows)
print(eval_df.to_string(index=False))

In [ ]:
best_k = 2

kmeans_model = KMeans(n_clusters=best_k, n_init=10, random_state=23)
kmeans_labels = kmeans_model.fit_predict(X_cluster_scaled)
country_avg['Cluster'] = kmeans_labels

sil, db, dunn_val, ch = evaluate_clustering(X_cluster_scaled, kmeans_labels)
print(f'k={best_k}: Silhouette={sil:.4f}, DB={db:.4f}, Dunn={dunn_val:.4f}, CH={ch:.2f}')
print(f'\nCluster distribution:')
print(country_avg['Cluster'].value_counts().sort_index())

## Cluster Visualizations

In [ ]:
# RF feature importance
rf_clf = RandomForestClassifier(n_estimators=100, random_state=23)
rf_clf.fit(X_cluster_scaled, kmeans_labels)

feature_importance_cl = pd.DataFrame({
    'Feature': cluster_features,
    'Importance': rf_clf.feature_importances_
}).sort_values('Importance', ascending=False)

top_10_features = feature_importance_cl.head(10)['Feature'].tolist()

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_cl.head(10)['Feature'].str[:40], feature_importance_cl.head(10)['Importance'])
plt.xlabel('Importance')
plt.title('Top 10 Features for Cluster Differentiation')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D scatter
def create_pca_scatterplot(X_scaled, X_clusters):
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    plt.figure(figsize=(7, 5))
    for cluster_id in np.unique(X_clusters):
        idx = X_clusters == cluster_id
        plt.scatter(X_pca[idx, 0], X_pca[idx, 1], s=40, label=f'Cluster {cluster_id}')
    plt.title('PCA 2D Scatterplot of Clusters')
    plt.xlabel('PC 1'); plt.ylabel('PC 2')
    plt.legend(title='Cluster ID')
    plt.show()

create_pca_scatterplot(X_cluster_scaled, kmeans_labels)

In [ ]:
# Radar chart
def create_radar_graph(X, X_clusters):
    clustered_df = X.copy()
    clustered_df['Cluster'] = X_clusters
    cluster_means = clustered_df.groupby('Cluster').mean()
    labels = cluster_means.columns
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]
    plt.figure(figsize=(8, 8))
    for i, row in cluster_means.iterrows():
        values = row.tolist() + row.tolist()[:1]
        plt.polar(angles, values, label=f'Cluster {i}')
    plt.xticks(angles[:-1], labels, fontsize=8)
    plt.title('Cluster Profiles (Feature Means)')
    plt.legend()
    plt.show()

cluster_means_raw = pd.DataFrame(X_cluster_scaled, columns=cluster_features)
cluster_means_raw['KMeans_Cluster'] = kmeans_labels
cluster_means_raw = cluster_means_raw.groupby('KMeans_Cluster')[top_10_features].mean()
scaler_radar = StandardScaler()
X_top10_scaled = pd.DataFrame(
    scaler_radar.fit_transform(cluster_means_raw),
    columns=cluster_means_raw.columns, index=cluster_means_raw.index
)
X_top10_scaled.columns = [c[:20] + '...' if len(c) > 20 else c for c in X_top10_scaled.columns]
create_radar_graph(X_top10_scaled, X_top10_scaled.index.values)

In [ ]:
radar_features = ['Agriculture, forestry, and fishing, value added (% of GDP)',
 'Population ages 0-14 (% of total population)',
 'Fixed broadband subscriptions (per 100 people)',
 'Mortality rate, under-5 (per 1,000 live births)',
 'Fixed telephone subscriptions (per 100 people)',
 'Prevalence of anemia among children (% of children ages 6-59 months)',
 'Life expectancy at birth, total (years)',
 'Price level ratio of PPP conversion factor (GDP) to market exchange rate',
 'Life expectancy at birth, female (years)',
 'Access to electricity (% of population)']

available = [f for f in radar_features if f in cluster_features]
cluster_means_raw2 = pd.DataFrame(X_cluster_scaled, columns=cluster_features)
cluster_means_raw2['KMeans_Cluster'] = kmeans_labels
cluster_means_raw2 = cluster_means_raw2.groupby('KMeans_Cluster')[available].mean()
scaler_radar2 = StandardScaler()
X_desc_scaled = pd.DataFrame(
    scaler_radar2.fit_transform(cluster_means_raw2),
    columns=cluster_means_raw2.columns, index=cluster_means_raw2.index
)
X_desc_scaled.columns = [c[:20] + '...' if len(c) > 20 else c for c in X_desc_scaled.columns]
create_radar_graph(X_desc_scaled, X_desc_scaled.index.values)

In [ ]:
[f for f in radar_features if f not in available]

In [ ]:
# Boxplots
def create_feature_boxplots(X, X_clusters):
    clustered_df = X.copy()
    clustered_df['Cluster'] = X_clusters
    for col in X.columns:
        clustered_df.boxplot(column=col, by='Cluster', figsize=(6, 4))
        plt.title(f'{col} by Cluster')
        plt.suptitle('')
        plt.show()

X_top5 = pd.DataFrame(X_cluster_scaled, columns=cluster_features)[radar_features[:5]].copy()
create_feature_boxplots(X_top5, kmeans_labels)

## Cluster Labeling

In [ ]:
# Map cluster labels to train_df
cluster_map = country_avg.set_index('Country Name')['Cluster'].to_dict()
train_df['Cluster'] = train_df['Country Name'].map(cluster_map)

print('Train clusters:')
print(train_df['Cluster'].value_counts().sort_index())

## Per-Cluster IQR Clipping

In [ ]:
# IQR clipping on TRAIN only
cluster_gni_std = train_df.groupby('Cluster')[target_col].std()
median_std = cluster_gni_std.median()

for cid in sorted(train_df['Cluster'].unique()):
    c_train = train_df['Cluster'] == cid
    mult = 1.0 if cluster_gni_std.get(cid, 0) > median_std else 1.5
    for col in feature_cols:
        Q1 = train_df.loc[c_train, col].quantile(0.25)
        Q3 = train_df.loc[c_train, col].quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - mult * IQR, Q3 + mult * IQR
        train_df.loc[c_train, col] = train_df.loc[c_train, col].clip(lo, hi)

print('IQR clipping applied per cluster')

## Multicollinearity Test (VIF) per Cluster

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

corr_threshold = 0.3

for cid in sorted(train_df["Cluster"].unique()):
    c_mask = train_df["Cluster"] == cid
    X_c = train_df.loc[c_mask, feature_cols]
    y_c = train_df.loc[c_mask, target_col]

    # Correlation filter (same as Lasso pipeline)
    c_corr = X_c.corrwith(y_c).abs().dropna()
    c_corr_feats = c_corr[c_corr >= corr_threshold].index.tolist()

    scaler_vif = StandardScaler()
    X_scaled = scaler_vif.fit_transform(X_c[c_corr_feats])

    vif_data = pd.DataFrame({
        "Feature": c_corr_feats,
        "VIF": [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
    }).sort_values("VIF", ascending=False)

    high_vif = (vif_data["VIF"] > 10).sum()
    max_vif = vif_data["VIF"].max()
    print(f"Cluster {cid}: {len(c_corr_feats)} corr features, "
          f"{high_vif} with VIF>10 (max VIF={max_vif:.1f})")
    print(vif_data.head(5).to_string(index=False))
    print()

## Feature Selection Functions

In [ ]:
LASSO_ALPHAS = [0.01, 0.1, 1, 10, 50, 100, 500, 1000]

def lasso_feature_select(X_c_all, y_c, feature_cols, corr_threshold=0.3, max_feat_pct=0.20):
    n_train = len(X_c_all)
    cv_folds = min(5, n_train // 2)
    c_corr = X_c_all[feature_cols].corrwith(pd.Series(y_c, index=X_c_all.index)).abs().dropna()
    c_corr_feats = c_corr[c_corr >= corr_threshold].index.tolist()
    if len(c_corr_feats) < 3:
        c_corr_feats = c_corr.nlargest(10).index.tolist()
    scaler_l = StandardScaler()
    X_l = scaler_l.fit_transform(X_c_all[c_corr_feats].values)
    lasso_grid = GridSearchCV(
        Lasso(max_iter=10000, random_state=23),
        {'alpha': LASSO_ALPHAS}, cv=cv_folds, scoring='r2', n_jobs=-1
    )
    lasso_grid.fit(X_l, y_c)
    best_lasso = lasso_grid.best_estimator_
    max_feats = max(3, int(n_train * max_feat_pct))
    selected = [f for f, c in zip(c_corr_feats, best_lasso.coef_) if c != 0]
    if len(selected) > max_feats:
        importance = np.abs(best_lasso.coef_)
        top_idx = np.argsort(importance)[::-1]
        selected = [c_corr_feats[i] for i in top_idx[:max_feats] if best_lasso.coef_[i] != 0]
    if len(selected) < 3:
        selected = c_corr.nlargest(5).index.tolist()
    return selected, lasso_grid.best_params_['alpha']

def f_regression_select(X_c_all, y_c, feature_cols, top_n=10):
    scaler_f = StandardScaler()
    X_f = scaler_f.fit_transform(X_c_all[feature_cols].values)
    f_scores, _ = f_regression(X_f, y_c)
    feat_df = pd.DataFrame({'feature': feature_cols, 'f_score': f_scores}).sort_values('f_score', ascending=False)
    n_select = min(top_n, max(3, int(len(X_c_all) * 0.20)))
    return feat_df.head(n_select)['feature'].tolist()

def auto_feature_select(X_c_all, y_c, feature_cols):
    """Try both lasso and f_regression, pick best by KNN CV R2."""
    cv_folds = min(5, len(X_c_all) // 2)
    quick_knn_params = {'n_neighbors': [3, 5, 7], 'weights': ['distance'], 'metric': ['manhattan']}

    feats_lasso, _ = lasso_feature_select(X_c_all, y_c, feature_cols)
    feats_f = f_regression_select(X_c_all, y_c, feature_cols)

    best_feats, best_score, best_method = feats_lasso, -999, 'lasso'
    for name, feats in [('lasso', feats_lasso), ('f_reg', feats_f)]:
        scaler = StandardScaler()
        X_s = scaler.fit_transform(X_c_all[feats].values)
        gs = GridSearchCV(KNeighborsRegressor(), quick_knn_params, cv=cv_folds, scoring='r2', n_jobs=-1)
        gs.fit(X_s, y_c)
        if gs.best_score_ > best_score:
            best_score = gs.best_score_
            best_feats = feats
            best_method = name
    return best_feats, best_method

## Per-Cluster KNN Training

In [ ]:
knn_params = {
    'n_neighbors': [2, 3, 4, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
}

cluster_models = {}
cluster_scalers = {}
cluster_selected_features = {}

### Cluster 0

In [ ]:
cid = 0
c_mask = train_df['Cluster'] == cid
X_c = train_df.loc[c_mask, feature_cols]
y_c = train_df.loc[c_mask, target_col].values
n_c = len(X_c)
cv_folds = min(5, n_c // 2)

# Auto feature selection
c_selected, fs_method = auto_feature_select(X_c, y_c, feature_cols)
cluster_selected_features[cid] = c_selected

# Scale features
knn_scaler = StandardScaler()
X_c_scaled = knn_scaler.fit_transform(X_c[c_selected].values)
cluster_scalers[cid] = knn_scaler

# GridSearchCV for KNN
knn_gs = GridSearchCV(
    KNeighborsRegressor(), knn_params,
    cv=cv_folds, scoring='r2', n_jobs=-1
)
knn_gs.fit(X_c_scaled, y_c)
cluster_models[cid] = knn_gs.best_estimator_

print(f'Cluster {cid} (n={n_c}, fs={fs_method}, feats={len(c_selected)})')
print(f'Features: {c_selected}')
print(f'KNN params: {knn_gs.best_params_}')
print(f'CV R2: {knn_gs.best_score_:.4f}')

### Cluster 1

In [ ]:
cid = 1
c_mask = train_df['Cluster'] == cid
X_c = train_df.loc[c_mask, feature_cols]
y_c = train_df.loc[c_mask, target_col].values
n_c = len(X_c)
cv_folds = min(5, n_c // 2)

# Auto feature selection
c_selected, fs_method = auto_feature_select(X_c, y_c, feature_cols)
cluster_selected_features[cid] = c_selected

# Scale features
knn_scaler = StandardScaler()
X_c_scaled = knn_scaler.fit_transform(X_c[c_selected].values)
cluster_scalers[cid] = knn_scaler

# GridSearchCV for KNN
knn_gs = GridSearchCV(
    KNeighborsRegressor(), knn_params,
    cv=cv_folds, scoring='r2', n_jobs=-1
)
knn_gs.fit(X_c_scaled, y_c)
cluster_models[cid] = knn_gs.best_estimator_

print(f'Cluster {cid} (n={n_c}, fs={fs_method}, feats={len(c_selected)})')
print(f'Features: {c_selected}')
print(f'KNN params: {knn_gs.best_params_}')
print(f'CV R2: {knn_gs.best_score_:.4f}')

## Prediction

In [ ]:
# Assign test clusters
X_test_cluster_feats = test_df[cluster_features].values
X_test_cluster_scaled = scaler_cluster.transform(X_test_cluster_feats)
test_df['Cluster'] = kmeans_model.predict(X_test_cluster_scaled)

print('Test clusters:')
print(test_df['Cluster'].value_counts().sort_index())

# Per-cluster predictions
y_pred_piecewise = np.full(len(test_df), np.nan)

for cid in sorted(test_df['Cluster'].unique()):
    t_mask = test_df['Cluster'] == cid
    c_feats = cluster_selected_features[cid]
    X_t = cluster_scalers[cid].transform(test_df.loc[t_mask, c_feats].values)
    y_pred_piecewise[t_mask.values] = cluster_models[cid].predict(X_t)

## Evaluation

In [ ]:
# Per-cluster metrics
rows = []
for cid in sorted(test_df['Cluster'].unique()):
    t_mask = test_df['Cluster'] == cid
    y_true_c = test_df.loc[t_mask, target_col].values
    y_pred_c = y_pred_piecewise[t_mask.values]
    n = t_mask.sum()

    rows.append({
        'Cluster': cid, 'N_test': n, 'N_features': len(cluster_selected_features[cid]),
        'R2': round(r2_score(y_true_c, y_pred_c), 4),
        'RMSE': round(root_mean_squared_error(y_true_c, y_pred_c), 2),
        'MAE': round(mean_absolute_error(y_true_c, y_pred_c), 2),
    })
per_cluster_df = pd.DataFrame(rows)
print(per_cluster_df.to_string(index=False))

In [ ]:
# Overall piecewise metrics
valid = ~np.isnan(y_pred_piecewise)
pw_results = {
    'R2': r2_score(y_test[valid], y_pred_piecewise[valid]),
    'RMSE': root_mean_squared_error(y_test[valid], y_pred_piecewise[valid]),
    'MAE': mean_absolute_error(y_test[valid], y_pred_piecewise[valid])
}

print(f'R2:   {pw_results["R2"]:.6f}')
print(f'RMSE: {pw_results["RMSE"]:.2f}')
print(f'MAE:  {pw_results["MAE"]:.2f}')

---

# Summarize

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Baseline: Corr+Lasso+RF', 'R2': round(bl_results['R2'], 4),
     'RMSE': round(bl_results['RMSE'], 2), 'MAE': round(bl_results['MAE'], 2)},
    {'Model': 'Single KNN (tuned)', 'R2': round(single_knn_results['R2'], 4),
     'RMSE': round(single_knn_results['RMSE'], 2), 'MAE': round(single_knn_results['MAE'], 2)},
    {'Model': 'Piecewise KNN (k=2)', 'R2': round(pw_results['R2'], 4),
     'RMSE': round(pw_results['RMSE'], 2), 'MAE': round(pw_results['MAE'], 2)},
])
print(comparison.to_string(index=False))

In [ ]:
r2_imp_rf = (pw_results['R2'] - bl_results['R2']) / abs(bl_results['R2']) * 100
rmse_imp_rf = (bl_results['RMSE'] - pw_results['RMSE']) / bl_results['RMSE'] * 100
r2_imp_knn = (pw_results['R2'] - single_knn_results['R2']) / abs(single_knn_results['R2']) * 100
rmse_imp_knn = (single_knn_results['RMSE'] - pw_results['RMSE']) / single_knn_results['RMSE'] * 100

print(f'Piecewise KNN vs Baseline RF:   R2 {r2_imp_rf:+.1f}%, RMSE {rmse_imp_rf:+.1f}%')
print(f'Piecewise KNN vs Single KNN:   R2 {r2_imp_knn:+.1f}%, RMSE {rmse_imp_knn:+.1f}%')

In [ ]:
# Actual vs Predicted side-by-side
from matplotlib.patches import Patch

all_test_cluster = sorted(test_df['Cluster'].unique())
colors_map = {cid: plt.cm.tab10.colors[i] for i, cid in enumerate(all_test_cluster)}
point_colors = [colors_map[c] for c in test_df['Cluster'].values]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, y_pred, title in [
    (axes[0], y_pred_baseline, f'Baseline RF (R2={bl_results["R2"]:.3f})'),
    (axes[1], y_pred_piecewise, f'Piecewise KNN (R2={pw_results["R2"]:.3f})'),
]:
    ax.scatter(y_test, y_pred, c=point_colors, alpha=0.7, s=40)
    lo = min(y_test.min(), y_pred.min())
    hi = max(y_test.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='Ideal')
    ax.set_xlabel('Actual GNI'); ax.set_ylabel('Predicted GNI')
    ax.set_title(title); ax.legend()

legend_el = [Patch(facecolor=colors_map[c], label=f'Cluster {c}') for c in all_test_cluster]
fig.legend(handles=legend_el, loc='lower center', ncol=len(all_test_cluster),
           bbox_to_anchor=(0.5, -0.05), title='Cluster')
plt.suptitle('Actual vs Predicted (Test 2021)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Per-cluster R2 and RMSE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
valid_rows = per_cluster_df.dropna(subset=['R2'])
bar_c = [colors_map.get(c, 'gray') for c in valid_rows['Cluster']]

axes[0].bar(valid_rows['Cluster'].astype(str), valid_rows['R2'], color=bar_c, edgecolor='black')
axes[0].axhline(bl_results['R2'], color='red', linestyle='--', label=f'Baseline={bl_results["R2"]:.3f}')
axes[0].set_title('Per-Cluster R2'); axes[0].set_xlabel('Cluster')
axes[0].set_ylim(0, 1.05); axes[0].legend()

valid_r = per_cluster_df.dropna(subset=['RMSE'])
bar_r = [colors_map.get(c, 'gray') for c in valid_r['Cluster']]
axes[1].bar(valid_r['Cluster'].astype(str), valid_r['RMSE'], color=bar_r, edgecolor='black')
axes[1].axhline(bl_results['RMSE'], color='red', linestyle='--', label=f'Baseline={bl_results["RMSE"]:.0f}')
axes[1].set_title('Per-Cluster RMSE'); axes[1].set_xlabel('Cluster'); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print(comparison.to_string(index=False))
print()
print('Per-Cluster (kmeans-assigned):')
print(per_cluster_df.to_string(index=False))